<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.6_Praktikum_Object_Detection.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.6: Praktikum Object Detection (Bounding Box, IoU, dan NMS)
**Tujuan Pembelajaran:**
1. Melakukan konversi format koordinat Bounding Box ($xyxy \leftrightarrow xywh$).
2. Mengimplementasikan kalkulasi vektorisasi Intersection over Union (IoU).
3. Membangun dan menguji algoritma Non-Maximum Suppression (NMS) untuk eliminasi deteksi ganda.

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("Modul praktikum deteksi objek siap!")

### Bagian 1: Konversi Koordinat dan Kalkulasi IoU

In [ ]:
def xywh_to_xyxy(boxes_xywh):
    xc, yc, w, h = boxes_xywh[:, 0], boxes_xywh[:, 1], boxes_xywh[:, 2], boxes_xywh[:, 3]
    x1 = xc - w / 2.0
    y1 = yc - h / 2.0
    x2 = xc + w / 2.0
    y2 = yc + h / 2.0
    return np.stack([x1, y1, x2, y2], axis=1)

def compute_single_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

# Uji IoU sampel
gt_box = np.array([10, 10, 50, 50], dtype=float)
pred_box = np.array([20, 20, 60, 60], dtype=float)
iou_val = compute_single_iou(gt_box, pred_box)
print(f"IoU antara GT [10, 10, 50, 50] dan Pred [20, 20, 60, 60]: {iou_val:.4f}")
assert abs(iou_val - (900.0 / 2300.0)) < 1e-4, "Kalkulasi IoU salah!"
print("[VALIDASI SUKSES] Formulasi matematis IoU teruji benar!")

### Bagian 2: Implementasi Algoritma Non-Maximum Suppression (NMS)

In [ ]:
def nms_pure_python(boxes, scores, threshold=0.5):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        iou = inter / (areas[i] + areas[order[1:]] - inter)
        
        inds = np.where(iou <= threshold)[0]
        order = order[inds + 1]
        
    return keep

# Simulasi beberapa prediksi kotak di sekitar 1 buah sawit
sim_boxes = np.array([
    [100, 100, 200, 200],  # Box 0: Sangat tepat (Conf 0.95)
    [105,  98, 202, 205],  # Box 1: Redundan tumpang tindih tinggi (Conf 0.88)
    [ 95, 102, 198, 195],  # Box 2: Redundan tumpang tindih tinggi (Conf 0.76)
    [300, 300, 380, 380]   # Box 3: Objek buah lain terpisah (Conf 0.91)
], dtype=float)

sim_scores = np.array([0.95, 0.88, 0.76, 0.91], dtype=float)

kept_indices = nms_pure_python(sim_boxes, sim_scores, threshold=0.5)
print(f"Kotak masukan: {len(sim_boxes)} kotak")
print(f"Kotak setelah NMS: {len(kept_indices)} kotak (Indeks terpilih: {kept_indices})")
assert 0 in kept_indices and 3 in kept_indices and len(kept_indices) == 2, "NMS gagal menapis redundansi!"
print("[SUKSES] NMS berhasil mempertahankan kotak unik dengan skor tertinggi!")

### Bagian 3: Visualisasi Hasil Deteksi Sebelum vs Sesudah NMS

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Sebelum NMS
ax1.set_xlim(50, 450)
ax1.set_ylim(450, 50) # Invert Y axis seperti citra
ax1.set_title(f'Sebelum NMS ({len(sim_boxes)} Kotak Redundan)')
for idx, b in enumerate(sim_boxes):
    rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=2,
                             edgecolor='red' if idx != 0 and idx != 3 else 'green', facecolor='none')
    ax1.add_patch(rect)
    ax1.text(b[0], b[1]-5, f"Conf: {sim_scores[idx]:.2f}", fontsize=8, color='black')
ax1.grid(True, linestyle='--', alpha=0.5)

# Sesudah NMS
ax2.set_xlim(50, 450)
ax2.set_ylim(450, 50)
ax2.set_title(f'Setelah NMS ({len(kept_indices)} Kotak Unik Bersih)')
for idx in kept_indices:
    b = sim_boxes[idx]
    rect = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], linewidth=2.5,
                             edgecolor='green', facecolor='none')
    ax2.add_patch(rect)
    ax2.text(b[0], b[1]-5, f"Tandan Sawit: {sim_scores[idx]:.2f}", fontsize=9, fontweight='bold', color='green')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('praktikum_nms_bounding_boxes_12_6.png', dpi=200)
plt.close()
print("[OK] Visualisasi NMS disimpan sebagai 'praktikum_nms_bounding_boxes_12_6.png'")